# Gemini App Chatting with PDFs

In [ ]:
from dotenv import load_dotenv

load_dotenv()

import os
import getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"var: ")

_set_env("GEMINI_API_KEY")

In [ ]:
import httpx
import pathlib

# Download the AgentBench paper PDF locally (portable — no wget dependency).
# Only fetch if it is not already present, so re-runs do not overwrite it.
pdf_path = pathlib.Path("./assets-resources/paper.pdf")
if not pdf_path.exists():
    pdf_path.write_bytes(httpx.get("https://arxiv.org/pdf/2308.03688", follow_redirects=True).content)
print(f"paper.pdf ready ({pdf_path.stat().st_size:,} bytes)")

In [ ]:
from google import genai
from google.genai import types
import os

gemini_api_key = os.getenv("GEMINI_API_KEY")

client = genai.Client(api_key=gemini_api_key)

# Read the PDF we downloaded in the cell above
doc_data = pdf_path.read_bytes()

prompt = "Summarize this document"
response = client.models.generate_content(
  model="gemini-3.8-flash",
  contents=[
      types.Part.from_bytes(
        data=doc_data,
        mime_type='application/pdf',
      ),
      prompt])
print(response.text)

In [ ]:
# Same local PDF, no second download
doc_data = pdf_path.read_bytes()

prompt = "Extract all the tables from this paper as json objects."
response = client.models.generate_content(
  model="gemini-3.8-flash",
  contents=[
      types.Part.from_bytes(
        data=doc_data,
        mime_type='application/pdf',
      ),
      prompt])
print(response.text)

![](./assets-resources/sample_image_from_paper_agent_bench.png)


In [ ]:
from google import genai

client = genai.Client(api_key=gemini_api_key)

my_file = client.files.upload(file="./assets-resources/sample_image_from_paper_agent_bench.png")

response = client.models.generate_content(
    model="gemini-3.8-flash",
    contents=[my_file, f"Explain this figure in a few bullet points."],
)

print(response.text)